# Аудит потери строк в `clustering_agg_v4_05.ipynb`

Этот ноутбук повторяет только подготовку данных из `clustering_agg_v4_05.ipynb` и проверяет гипотезу: строки пропали из-за фильтра `SUM > 0`, то есть потому что по всем колонкам `TAG_` сумма равна `0`.

Важно: версия `_05` читает только `DATA_CSV_PART1`. В `clustering_agg_v4.ipynb` есть еще `inner merge` с `DATA_CSV_PART2`, это отдельный источник возможной потери строк.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from config import DATA_CSV_PART1, TARGET_COL

KEY_COL = 'POLICY_ZV'
TAG_JOIN_COL = 'TAG_JOIN_IND'

OUTPUT_PATH = Path('artifacts/csv/all_groups_agg_coef.csv')
AUDIT_DIR = Path('artifacts/audit_v4_05')
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CSV_PART1, OUTPUT_PATH, AUDIT_DIR

## 1. Читаем исходный файл как в `_05`

In [ ]:
raw = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
raw['_source_row_number'] = np.arange(len(raw))

print(f'Строк в исходном DATA_CSV_PART1: {len(raw):,}')
print(f'Колонок: {raw.shape[1]:,}')
display(raw.head())

In [ ]:
required_cols = [KEY_COL, 'CLAIMS_PART_DAM_COUNT']
missing_required = [col for col in required_cols if col not in raw.columns]
if missing_required:
    raise KeyError(f'Нет обязательных колонок: {missing_required}')

raw_key_stats = pd.DataFrame({
    'metric': [
        'rows',
        'non_null_POLICY_ZV',
        'unique_POLICY_ZV',
        'duplicate_POLICY_ZV_rows',
        'missing_POLICY_ZV',
    ],
    'value': [
        len(raw),
        raw[KEY_COL].notna().sum(),
        raw[KEY_COL].nunique(dropna=True),
        len(raw) - raw[KEY_COL].nunique(dropna=True),
        raw[KEY_COL].isna().sum(),
    ],
})
display(raw_key_stats)

## 2. Повторяем подготовку из `clustering_agg_v4_05.ipynb` по шагам

In [ ]:
def key_series(df: pd.DataFrame) -> pd.Series:
    if KEY_COL in df.columns:
        return df[KEY_COL]
    return df.index.to_series()


def step_stats(df: pd.DataFrame, step: str) -> dict:
    keys = key_series(df)
    rows = len(df)
    unique_keys = keys.nunique(dropna=True)
    non_null_keys = keys.notna().sum()
    return {
        'step': step,
        'rows': rows,
        'non_null_POLICY_ZV': non_null_keys,
        'unique_POLICY_ZV': unique_keys,
        'duplicate_POLICY_ZV_rows': rows - unique_keys,
        'missing_POLICY_ZV': rows - non_null_keys,
    }


steps = []
steps.append(step_stats(raw, '01 raw DATA_CSV_PART1'))

work = raw.copy()
work[TARGET_COL] = work['CLAIMS_PART_DAM_COUNT'].astype(bool).astype(int)
steps.append(step_stats(work, '02 after target conversion'))

work = work.set_index(KEY_COL)
steps.append(step_stats(work, '03 after set_index(POLICY_ZV)'))

if TAG_JOIN_COL in work.columns:
    work = work.drop(columns=[TAG_JOIN_COL])
steps.append(step_stats(work, '04 after drop TAG_JOIN_IND'))

tag_cols = work.filter(like='TAG_').columns.tolist()
if not tag_cols:
    raise ValueError('Не найдены колонки TAG_. Фильтр SUM > 0 удалит все строки.')

work['SUM'] = work[tag_cols].fillna(0).sum(axis=1)
work['NONZERO_TAG_COUNT'] = work[tag_cols].fillna(0).ne(0).sum(axis=1)
steps.append(step_stats(work, '05 after SUM calculation'))

zero_sum_mask = work['SUM'].le(0)
with_tags = work.loc[~zero_sum_mask].copy()
lost_zero_sum = work.loc[zero_sum_mask].copy()
steps.append(step_stats(with_tags, '06 after filter SUM > 0'))

steps_df = pd.DataFrame(steps)
steps_df['rows_lost_from_prev'] = steps_df['rows'].shift(1).sub(steps_df['rows']).fillna(0).astype(int)
steps_df['rows_lost_from_raw'] = len(raw) - steps_df['rows']
steps_df.to_csv(AUDIT_DIR / 'row_count_by_step.csv', index=False, encoding='utf-8-sig')

print(f'Колонок TAG_, участвующих в SUM: {len(tag_cols):,}')
display(steps_df)

## 3. Проверяем, правда ли исчезли строки с `SUM == 0`

In [ ]:
loss_from_filter = len(work) - len(with_tags)
zero_sum_rows = int(zero_sum_mask.sum())
all_tag_values_zero_rows = int((work['NONZERO_TAG_COUNT'] == 0).sum())
zero_sum_but_nonzero_tags = work.loc[zero_sum_mask & work['NONZERO_TAG_COUNT'].gt(0)].copy()

check_df = pd.DataFrame({
    'metric': [
        'rows_before_filter',
        'rows_after_SUM_gt_0_filter',
        'rows_lost_on_filter',
        'rows_with_SUM_le_0',
        'rows_where_all_TAG_values_are_0',
        'rows_with_SUM_le_0_but_some_TAG_nonzero',
        'loss_equals_SUM_le_0',
    ],
    'value': [
        len(work),
        len(with_tags),
        loss_from_filter,
        zero_sum_rows,
        all_tag_values_zero_rows,
        len(zero_sum_but_nonzero_tags),
        loss_from_filter == zero_sum_rows,
    ],
})
check_df.to_csv(AUDIT_DIR / 'sum_zero_check.csv', index=False, encoding='utf-8-sig')
display(check_df)

assert loss_from_filter == zero_sum_rows, 'Потеря строк на фильтре не равна количеству строк с SUM <= 0'
print('OK: фильтр SUM > 0 удалил ровно строки с SUM <= 0.')

In [ ]:
sum_distribution = (
    work['SUM']
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis('SUM')
    .reset_index(name='rows')
)
sum_distribution['share'] = sum_distribution['rows'] / len(work)
sum_distribution.to_csv(AUDIT_DIR / 'sum_distribution.csv', index=False, encoding='utf-8-sig')

display(work['SUM'].describe().to_frame('SUM'))
display(sum_distribution.head(30))

## 4. Сохраняем примеры удаленных строк

In [ ]:
lost_zero_sum_reset = lost_zero_sum.reset_index()
basic_cols = [
    KEY_COL,
    '_source_row_number',
    'CLAIMS_PART_DAM_COUNT',
    TARGET_COL,
    'SUM',
    'NONZERO_TAG_COUNT',
]
basic_cols = [col for col in basic_cols if col in lost_zero_sum_reset.columns]

lost_zero_sum_reset[basic_cols].head(1000).to_csv(
    AUDIT_DIR / 'lost_rows_sum_le_0_sample.csv',
    index=False,
    encoding='utf-8-sig',
)

if len(zero_sum_but_nonzero_tags) > 0:
    zero_sum_but_nonzero_tags.reset_index().head(1000).to_csv(
        AUDIT_DIR / 'sum_le_0_but_some_tags_nonzero_sample.csv',
        index=False,
        encoding='utf-8-sig',
    )

print(f'Удаленных строк с SUM <= 0: {len(lost_zero_sum_reset):,}')
print(f'Сэмпл сохранен: {AUDIT_DIR / "lost_rows_sum_le_0_sample.csv"}')
display(lost_zero_sum_reset[basic_cols].head(20))

## 5. Сравниваем ожидаемые строки с уже сохраненным итоговым CSV

Этот блок проверяет `artifacts/csv/all_groups_agg_coef.csv`, если файл уже создан запуском `clustering_agg_v4_05.ipynb`. Сравнение идет по количеству строк на каждый `POLICY_ZV`, поэтому дубликаты ключей тоже видны.

In [ ]:
def normalize_output_key_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if KEY_COL in df.columns:
        return df
    unnamed_cols = [col for col in df.columns if str(col).startswith('Unnamed')]
    if unnamed_cols:
        return df.rename(columns={unnamed_cols[0]: KEY_COL})
    raise KeyError(f'В итоговом CSV не найдена колонка {KEY_COL} или Unnamed index column')


if OUTPUT_PATH.exists():
    output = pd.read_csv(OUTPUT_PATH, encoding='utf-8-sig')
    output = normalize_output_key_column(output)

    expected_counts = with_tags.reset_index().groupby(KEY_COL, dropna=False).size().rename('expected_rows')
    output_counts = output.groupby(KEY_COL, dropna=False).size().rename('output_rows')
    compare_counts = pd.concat([expected_counts, output_counts], axis=1).fillna(0).astype(int)
    compare_counts['delta_output_minus_expected'] = compare_counts['output_rows'] - compare_counts['expected_rows']

    missing_in_output = compare_counts.loc[compare_counts['delta_output_minus_expected'] < 0].copy()
    extra_in_output = compare_counts.loc[compare_counts['delta_output_minus_expected'] > 0].copy()

    compare_summary = pd.DataFrame({
        'metric': [
            'expected_rows_after_SUM_gt_0',
            'output_rows',
            'row_delta_output_minus_expected',
            'POLICY_ZV_with_too_few_output_rows',
            'POLICY_ZV_with_extra_output_rows',
            'output_matches_expected_counts',
        ],
        'value': [
            len(with_tags),
            len(output),
            len(output) - len(with_tags),
            len(missing_in_output),
            len(extra_in_output),
            len(missing_in_output) == 0 and len(extra_in_output) == 0,
        ],
    })

    compare_summary.to_csv(AUDIT_DIR / 'output_compare_summary.csv', index=False, encoding='utf-8-sig')
    missing_in_output.reset_index().to_csv(AUDIT_DIR / 'output_missing_expected_policy_zv.csv', index=False, encoding='utf-8-sig')
    extra_in_output.reset_index().to_csv(AUDIT_DIR / 'output_extra_policy_zv.csv', index=False, encoding='utf-8-sig')

    display(compare_summary)
    display(missing_in_output.head(20))
    display(extra_in_output.head(20))
else:
    print(f'Файл {OUTPUT_PATH} не найден. Сначала запустите clustering_agg_v4_05.ipynb или пропустите этот блок.')

## 6. Дополнительная проверка: какие агрегированные сырые скоры были бы нулевыми без фильтра

Фильтр в `_05` смотрит на все `TAG_` колонки, а не только на TAG из трех агрегатов. Поэтому строка может остаться в расчете, даже если все три агрегированных скора равны `0`, но есть какой-то другой `TAG_`.

In [ ]:
from config import FINAL_WEIGHTS, TAGS_EXCEL_PATH, load_tag_lists

raw_score_check = pd.DataFrame(index=work.index)

for group_name in ['auto_lover', 'shopping']:
    weights = pd.Series(FINAL_WEIGHTS[group_name], dtype=float)
    X_group = work.reindex(columns=weights.index, fill_value=0).fillna(0).astype(float)
    raw_score_check[f'{group_name}_raw_score'] = X_group.mul(weights, axis=1).sum(axis=1)

try:
    tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
    alcohol_tags = load_tag_lists(tags_descriptions)['alcohol_features_list']
    raw_score_check['alcohol_raw_score'] = (
        work.reindex(columns=alcohol_tags, fill_value=0).fillna(0).astype(float).sum(axis=1)
    )
except Exception as exc:
    print(f'Не удалось посчитать alcohol_raw_score через Excel-справочник: {exc}')

score_cols = raw_score_check.columns.tolist()
raw_score_check['all_available_raw_scores_zero'] = raw_score_check[score_cols].fillna(0).eq(0).all(axis=1)
raw_score_check['SUM_le_0'] = zero_sum_mask
raw_score_check['kept_by_SUM_gt_0_filter'] = ~zero_sum_mask

score_zero_crosstab = pd.crosstab(
    raw_score_check['SUM_le_0'],
    raw_score_check['all_available_raw_scores_zero'],
    rownames=['SUM <= 0'],
    colnames=['all available raw scores == 0'],
)

raw_score_check.reset_index().to_csv(AUDIT_DIR / 'raw_score_zero_check.csv', index=False, encoding='utf-8-sig')
display(score_zero_crosstab)
display(raw_score_check[score_cols + ['SUM_le_0', 'kept_by_SUM_gt_0_filter']].describe(include='all'))

## Вывод, который надо смотреть

- `row_count_by_step.csv`: где именно поменялось число строк.
- `sum_zero_check.csv`: равна ли потеря строк количеству `SUM <= 0`.
- `lost_rows_sum_le_0_sample.csv`: примеры строк, которые удалил фильтр.
- `output_compare_summary.csv`: совпадает ли сохраненный итоговый файл с ожидаемыми строками после фильтра.